# Pipelines de procesamiento para el dataset de cosméticos

Este notebook construye un pipeline completo de limpieza y transformación para el dataset de `data/raw/products.csv`.

In [ ]:
# Configuración para ejecutar en Google Colab o local y definir clases de transformadores localmente
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn import set_config
from sklearn.base import BaseEstimator, TransformerMixin

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('Google Colab detectado. Sube `products.csv` cuando se te pida.')
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        data_filename = list(uploaded.keys())[0]
    else:
        raise FileNotFoundError('Debe subir el archivo `products.csv` en Colab antes de ejecutar el notebook.')
else:
    data_filename = '../data/raw/products.csv'

print('Usando ruta de datos:', data_filename)


class DropColumnsTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns_to_drop):
        self.columns_to_drop = columns_to_drop

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        cols = [col for col in self.columns_to_drop if col in X_copy.columns]
        return X_copy.drop(columns=cols)


class UnknownToNaNTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.replace('unknown', np.nan)


class DropHighMissingTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.8):
        self.threshold = threshold
        self.cols_to_drop_ = []

    def fit(self, X, y=None):
        pct_nulos = X.isnull().mean()
        self.cols_to_drop_ = pct_nulos[pct_nulos > self.threshold].index.tolist()
        return self

    def transform(self, X):
        X_copy = X.copy()
        cols = [c for c in self.cols_to_drop_ if c in X_copy.columns]
        return X_copy.drop(columns=cols)


class OutlierCapper(BaseEstimator, TransformerMixin):
    def __init__(self, apply_capping=True):
        self.apply_capping = apply_capping
        self.bounds_ = {}

    def fit(self, X, y=None):
        if not self.apply_capping:
            return self
        for col in X.select_dtypes(include=['number']).columns:
            Q1 = X[col].quantile(0.25)
            Q3 = X[col].quantile(0.75)
            IQR = Q3 - Q1
            self.bounds_[col] = (Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
        return self

    def transform(self, X):
        X_copy = X.copy()
        if not self.apply_capping:
            return X_copy
        for col, (lower, upper) in self.bounds_.items():
            if col in X_copy.columns:
                X_copy[col] = np.clip(X_copy[col], lower, upper)
        return X_copy


class DropZeroVarianceTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.cols_to_drop_ = []

    def fit(self, X, y=None):
        num_cols = X.select_dtypes(include=['number']).columns
        zero_variance = [col for col in num_cols if X[col].std() == 0]
        if len(num_cols) - len(zero_variance) == 0:
            self.cols_to_drop_ = []
        else:
            self.cols_to_drop_ = zero_variance
        return self

    def transform(self, X):
        X_copy = X.copy()
        cols = [c for c in self.cols_to_drop_ if c in X_copy.columns]
        return X_copy.drop(columns=cols)


class SmartImputerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, low_threshold=0.10):
        self.low_threshold = low_threshold
        self.cols_simples_ = []
        self.cols_complejas_ = []

    def fit(self, X, y=None):
        porcentaje_nulos = X.isnull().mean()
        for col in X.columns:
            pct = porcentaje_nulos[col]
            if 0 < pct <= self.low_threshold:
                self.cols_simples_.append(col)
            elif pct > self.low_threshold:
                self.cols_complejas_.append(col)
        print(f"SmartImputer - Simples (<10%): {self.cols_simples_}")
        print(f"SmartImputer - Complejas (>10%): {self.cols_complejas_} (PENDIENTE)")
        return self

    def transform(self, X):
        X_copy = X.copy()
        for col in self.cols_simples_:
            if pd.api.types.is_numeric_dtype(X_copy[col]):
                X_copy[col] = X_copy[col].fillna(X_copy[col].median())
            else:
                X_copy[col] = X_copy[col].fillna(X_copy[col].mode()[0])
        if self.cols_complejas_:
            for col in self.cols_complejas_:
                if pd.api.types.is_numeric_dtype(X_copy[col]):
                    X_copy[col] = X_copy[col].fillna(X_copy[col].median())
                else:
                    X_copy[col] = X_copy[col].fillna(X_copy[col].mode()[0])
        return X_copy

print('✅ Entorno y transformadores definidos localmente para ejecución independiente.')

In [ ]:
# Cargamos los datos crudos desde el archivo seleccionado o desde la ruta local
ruta = data_filename

df_raw = pd.read_csv(ruta, na_values=['NULL', 'null', ''])

# Aseguramos los tipos correctos para variables numéricas
for col in ['price_site', 'view_count', 'rating']:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

print('Dimensiones originales:', df_raw.shape)
print('\nColumnas del dataset:')
print(df_raw.columns.tolist())

In [ ]:
# Definimos qué columnas van por qué ruta
variables_numericas = ['rating', 'price_site', 'view_count']
variables_categoricas = ['brand', 'product_type']

print('Variables numéricas:', variables_numericas)
print('Variables categóricas:', variables_categoricas)

In [ ]:
# 1. Ruta para números (Capping con interruptor, Varianza Cero, Escalar)
num_pipe = Pipeline([
    ('capper', OutlierCapper(apply_capping=True)),
    ('zero_variance', DropZeroVarianceTransformer()),
    ('scaler', StandardScaler())
])

# 2. Ruta para textos (Solo codificar, ya que SmartImputer imputará antes)
cat_pipe = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. El enrutador
preprocesador = ColumnTransformer(
    transformers=[
        ('num', num_pipe, variables_numericas),
        ('cat', cat_pipe, variables_categoricas)
    ], remainder='drop'
)

# 4. El súper pipeline
pipeline_ml_completo = Pipeline([
    ('drop_leaks', DropColumnsTransformer(columns_to_drop=[
        'ID', 'img', 'name', 'description', 'shade_img', 'dupes'
    ])),
    ('limpieza_unknowns', UnknownToNaNTransformer()),
    ('drop_high_nan', DropHighMissingTransformer(threshold=0.8)),
    ('smart_imputer', SmartImputerTransformer(low_threshold=0.10)),
    ('preprocesamiento', preprocesador)
])

set_config(display='diagram')
pipeline_ml_completo

In [ ]:
# Ejecución del pipeline
print('Dimensiones originales:', df_raw.shape)
df_procesado = pipeline_ml_completo.fit_transform(df_raw)
print('Dimensiones procesadas:', df_procesado.shape)

print(f'\nTipo de dato de salida: {type(df_procesado)}')
print('\nPrimeras 3 filas de la matriz lista para el modelo:')
print(df_procesado[:3])

In [ ]:
# Recuperar nombres finales de variables para el modelo
enrutador = pipeline_ml_completo.named_steps['preprocesamiento']

if hasattr(enrutador, 'get_feature_names_out'):
    nombres_finales = enrutador.get_feature_names_out()
    print(f'Total de variables finales listas para el modelo: {len(nombres_finales)}\n')
    for i, nombre in enumerate(nombres_finales):
        nombre_limpio = nombre.replace('num__', '').replace('cat__', '')
        print(f'{i+1:02d}. {nombre_limpio}')
else:
    print('El ColumnTransformer no expone get_feature_names_out en esta versión de sklearn.')

## Resumen del pipeline contextualizado

- Se eliminan columnas de texto y metadatos no útiles para el modelo.
- Se convierten valores desconocidos en `NaN` y se eliminan columnas con mucho missing.
- Se imputa de forma inteligente usando estrategias diferentes para variables numéricas y categóricas.
- Se aplica capping a outliers en variables numéricas y se escalan con `StandardScaler`.
- Las variables categóricas `brand` y `product_type` se codifican con OneHotEncoder.

Este flujo está adaptado al dataset real de cosméticos y es un buen punto de partida para la siguiente fase de modelado.